# Subtask 1 — Test Inference & Submission (Notebook 5 of 5)

**Goal:** Generate the 800 test prediction PNGs, package them into a CodaBench ZIP, validate the ZIP, and provide a final visual spot-check before VB submits.

**HITL gate (final):** VB reviews the spot-check images and the validator output, then submits.

**Instructions:**
1. Run Notebook 4 first and note the best config values in `results/subtask1/ensemble/best_config.json`.
2. Edit the CONFIG cell below with those values (or leave defaults to use U-Net alone).
3. Run all cells top-to-bottom.
4. Inspect the spot-check plot.
5. Confirm the validator prints `OK` before submitting to CodaBench.

## 0. CONFIG — **Edit this cell before running**

In [ ]:
# ─── Edit these values based on Notebook 4 output ────────────────────────────
ALPHA              = 0.5       # 0.0 = pixel only, 1.0 = U-Net only
MEDIAN_FILTER_SIZE = 3         # None or 3
USE_TTA            = True      # test-time augmentation
TEMPORAL_OPTION    = "summary" # must match Notebook 3 training
PIXEL_MODEL_NAME   = "hgbc"    # 'hgbc' or 'etc'
# ─────────────────────────────────────────────────────────────────────────────

import csv
import json
import subprocess
import zipfile
from datetime import date
from pathlib import Path
from urllib.parse import urljoin
from urllib.request import urlopen
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import median_filter
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib

DATA_DIR    = None
LABEL_NAME  = "viticulture"
HF_BASE     = "https://huggingface.co/datasets/m-sakka/agripotential/resolve/main/"
CKPT_DIR    = Path("../results/subtask1/checkpoints")
SUBMIT_DIR  = Path("../results/subtask1/submissions")
MASKS_DIR   = Path("../results/subtask1/test_masks")
for d in [SUBMIT_DIR, MASKS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_BANDS = 10
CLASS_NAMES  = ["Very Low","Low","Medium","High","Very High"]
CLASS_COLORS = ["#d73027","#fc8d59","#fee08b","#91cf60","#1a9850"]

print(f"Device: {DEVICE}")
print(f"Config: alpha={ALPHA}, median={MEDIAN_FILTER_SIZE}, tta={USE_TTA}, temporal={TEMPORAL_OPTION}")

## 1. Load Metadata & Test Split

In [ ]:
import rasterio
from rasterio.windows import Window

def ref(name):
    if DATA_DIR:
        return str(Path(DATA_DIR) / name)
    return urljoin(HF_BASE, name)

def read_csv(name):
    r = ref(name)
    if r.startswith("http"):
        with urlopen(r, timeout=60) as h:
            lines = [l.decode("utf-8") for l in h]
    else:
        lines = Path(r).read_text().splitlines()
    return list(csv.DictReader(lines))

metadata  = read_csv("metadata.csv")
test_rows = read_csv("test.csv")

meta_df = pd.DataFrame(metadata)
meta_df["date"] = pd.to_datetime(meta_df[["year","month","day"]].astype(int))
meta_df = meta_df.sort_values("date").reset_index(drop=True)
sentinel_files = list(meta_df["filename"])
n_times = len(sentinel_files)
tf_months = list(meta_df["month"].astype(int))
SPRING_IDX = [i for i,m in enumerate(tf_months) if m in (3,4,5)]
SUMMER_IDX = [i for i,m in enumerate(tf_months) if m in (6,7,8)]
AUTUMN_IDX = [i for i,m in enumerate(tf_months) if m in (9,10,11)]
WINTER_IDX = [i for i,m in enumerate(tf_months) if m in (12,1,2)]

print(f"Test patches: {len(test_rows)}  (expect 800)")
print(f"Timeframes:   {n_times}")

## 2. Load Models

In [ ]:
# Pixel model
pixel_model = None
if ALPHA < 1.0:
    pixel_model = joblib.load(CKPT_DIR / f"{PIXEL_MODEL_NAME}_pixel.pkl")
    print(f"Loaded pixel model: {PIXEL_MODEL_NAME}")

# U-Net
unet = None
NORM_MEAN = NORM_STD = None
n_channels = None

class ConvBlock(nn.Module):
    def __init__(self,i,o):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(i,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(True),
            nn.Conv2d(o,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(self,x): return self.net(x)

class UNet(nn.Module):
    def __init__(self,in_ch,nc,b=32):
        super().__init__()
        self.e1=ConvBlock(in_ch,b); self.e2=ConvBlock(b,b*2)
        self.e3=ConvBlock(b*2,b*4); self.bot=ConvBlock(b*4,b*8)
        self.u3=nn.ConvTranspose2d(b*8,b*4,2,2); self.d3=ConvBlock(b*8,b*4)
        self.u2=nn.ConvTranspose2d(b*4,b*2,2,2); self.d2=ConvBlock(b*4,b*2)
        self.u1=nn.ConvTranspose2d(b*2,b,2,2);   self.d1=ConvBlock(b*2,b)
        self.h=nn.Conv2d(b,nc,1); self.p=nn.MaxPool2d(2)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p(e1)); e3=self.e3(self.p(e2))
        b=self.bot(self.p(e3))
        return self.h(self.d1(torch.cat([self.u1(self.d2(torch.cat([self.u2(self.d3(torch.cat([self.u3(b),e3],1))),e2],1))),e1],1)))

if ALPHA > 0.0:
    ckpt_path = CKPT_DIR / f"unet_{TEMPORAL_OPTION}_best.pt"
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    n_channels = ckpt["n_channels"]
    NORM_MEAN = np.array(ckpt["norm_mean"])
    NORM_STD  = np.array(ckpt["norm_std"])
    unet = UNet(n_channels, 5).to(DEVICE)
    unet.load_state_dict(ckpt["model_state"])
    unet.eval()
    print(f"Loaded U-Net from epoch {ckpt['epoch']} (val Acc±1={ckpt['val_acc_pm1']:.4f})")

## 3. Inference Loop

In [ ]:
def load_stack(row_idx, col_idx, patch_size):
    win = Window(col_idx, row_idx, patch_size, patch_size)
    return np.stack([
        (__import__('rasterio').open(ref(f)).read(window=win).astype(np.float32))
        for f in sentinel_files
    ], axis=0)

def stack_to_unet_tensor(stack):
    T, B, H, W = stack.shape
    if TEMPORAL_OPTION == "concat":
        x = stack.reshape(T*B, H, W)
    elif TEMPORAL_OPTION == "summary":
        x = np.concatenate([stack.mean(0), stack.std(0)], axis=0)
    elif TEMPORAL_OPTION == "seasonal":
        parts = []
        for idx in [SPRING_IDX, SUMMER_IDX, AUTUMN_IDX, WINTER_IDX]:
            parts.append(stack[idx].mean(0) if idx else np.zeros((B,H,W), dtype=np.float32))
        x = np.concatenate(parts, axis=0)
    t = torch.from_numpy(x)
    mean_t = torch.tensor(NORM_MEAN[:, None, None], dtype=torch.float32)
    std_t  = torch.tensor(NORM_STD[:, None, None],  dtype=torch.float32)
    return ((t - mean_t) / std_t).unsqueeze(0).to(DEVICE)

def pixel_proba(stack):
    T, B, H, W = stack.shape
    P = H * W
    pix = stack.reshape(T, B, P).transpose(2, 0, 1)
    raw = stack.transpose(2,3,0,1).reshape(P, T*B)
    tmean=pix.mean(1); tstd=pix.std(1); tmin=pix.min(1); tmax=pix.max(1)
    parts = [raw, tmean, tstd, tmin, tmax]
    for idx in [SPRING_IDX, SUMMER_IDX, AUTUMN_IDX, WINTER_IDX]:
        parts.append(pix[:,idx,:].mean(1) if idx else np.zeros((P,B),dtype=np.float32))
    nir=pix[:,:,7]; red=pix[:,:,3]
    ndvi=(nir-red)/(nir+red+1e-6)
    parts+=[ndvi.mean(1,keepdims=True), ndvi.std(1,keepdims=True)]
    feats = np.concatenate(parts, axis=1)
    probs = pixel_model.predict_proba(feats)  # (P, 5)
    return probs.T.reshape(5, H, W).astype(np.float32)

def unet_proba(stack, tta):
    x = stack_to_unet_tensor(stack)
    with torch.no_grad():
        logits = [unet(x)]
        if tta:
            logits.append(torch.flip(unet(torch.flip(x,[-1])),[-1]))
            logits.append(torch.flip(unet(torch.flip(x,[-2])),[-2]))
            logits.append(torch.flip(unet(torch.flip(x,[-1,-2])),[-1,-2]))
        probs = F.softmax(torch.stack(logits).mean(0), dim=1)[0].cpu().numpy()
    return probs  # (5, H, W)


print(f"Running inference on {len(test_rows)} test patches...")
class_hist = Counter()

for patch in tqdm(test_rows, desc="Test inference"):
    patch_id = str(patch["patch_id"])
    r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])
    stack = load_stack(r, c, ps)

    if ALPHA == 0.0:
        probs = pixel_proba(stack)
    elif ALPHA == 1.0:
        probs = unet_proba(stack, USE_TTA)
    else:
        probs = (1 - ALPHA) * pixel_proba(stack) + ALPHA * unet_proba(stack, USE_TTA)

    pred = probs.argmax(axis=0).astype(np.uint8)  # (H, W)

    if MEDIAN_FILTER_SIZE:
        pred = median_filter(pred, size=MEDIAN_FILTER_SIZE).astype(np.uint8)

    # Clamp to valid range (safety)
    pred = np.clip(pred, 0, 4).astype(np.uint8)
    class_hist.update(pred.ravel().tolist())

    # Save as greyscale PNG
    Image.fromarray(pred, mode="L").save(MASKS_DIR / f"{patch_id}.png")

print(f"\nSaved {len(test_rows)} masks to {MASKS_DIR}")

## 4. Sanity Check — Class Histogram

In [ ]:
total_px = sum(class_hist.values())
print("Predicted class distribution over all 800 test patches:")
for k in range(5):
    n = class_hist.get(k, 0)
    bar = "█" * int(40 * n / total_px)
    print(f"  Class {k} ({CLASS_NAMES[k]:>12s}): {n:>8,d}  ({100*n/total_px:5.1f}%)  {bar}")

missing_classes = [k for k in range(5) if class_hist.get(k, 0) == 0]
if missing_classes:
    print(f"\nWARNING: Classes {missing_classes} are completely absent from predictions.")
    print("This may indicate a model issue — inspect before submitting.")
else:
    print("\nAll 5 classes present in predictions. ✓")

## 5. Package ZIP

In [ ]:
run_date = date.today().strftime("%Y%m%d")
zip_path = SUBMIT_DIR / f"submit_{run_date}.zip"

mask_files = sorted(MASKS_DIR.glob("*.png"))
print(f"Packing {len(mask_files)} PNG masks into {zip_path}...")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for mf in mask_files:
        zf.write(mf, arcname=mf.name)  # no subfolders

print(f"ZIP size: {zip_path.stat().st_size / 1e6:.1f} MB")
print(f"Saved → {zip_path}")

## 6. Validate ZIP

In [ ]:
# Run the project validator — must exit 0 before submitting.
# Requires test.csv to be present locally for --expected-ids-file check.
test_csv_path = Path(DATA_DIR) / "test.csv" if DATA_DIR else None
validator_script = Path("../scripts/validate_submission_zip.py")

cmd = [
    "python", str(validator_script),
    "--zip-path", str(zip_path),
    "--subtask1-codabench",
    "--expected-count", "800",
    "--check-class-values",
]
if test_csv_path and test_csv_path.exists():
    cmd += ["--expected-ids-file", str(test_csv_path)]
else:
    print("Note: test.csv not found locally — skipping patch_id cross-check.")

print("Running validator:")
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
if result.returncode == 0:
    print("\n✓ Validator passed — ZIP is ready for CodaBench submission.")
else:
    print(f"\n✗ Validator failed (exit code {result.returncode}) — fix before submitting!")

## 7. Visual Spot-Check (5 random test patches)

In [ ]:
# No ground truth for test; we just verify predictions look reasonable.
import random
random.seed(0)
spot_patches = random.sample(test_rows, min(5, len(test_rows)))

cmap = mcolors.ListedColormap(CLASS_COLORS)
norm = mcolors.BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5], cmap.N)
VIS_RGB = [3, 2, 1]  # B4, B3, B2

fig, axes = plt.subplots(len(spot_patches), 2, figsize=(8, 3 * len(spot_patches)))
if len(spot_patches) == 1:
    axes = [axes]

for row_i, patch in enumerate(spot_patches):
    patch_id = str(patch["patch_id"])
    r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])

    # Last Sentinel-2 timeframe as RGB
    with rasterio.open(ref(sentinel_files[-1])) as src:
        rgb_raw = src.read(VIS_RGB, window=Window(c, r, ps, ps)).astype(np.float32)
    rgb = np.clip(rgb_raw / (np.percentile(rgb_raw, 98) + 1e-6), 0, 1).transpose(1, 2, 0)

    pred = np.array(Image.open(MASKS_DIR / f"{patch_id}.png"))

    axes[row_i][0].imshow(rgb)
    axes[row_i][0].set_title(f"RGB (last frame)  {patch_id}", fontsize=8)
    axes[row_i][0].axis("off")
    axes[row_i][1].imshow(pred, cmap=cmap, norm=norm, interpolation="nearest")
    axes[row_i][1].set_title(f"Predicted mask", fontsize=8)
    axes[row_i][1].axis("off")

# Add colour legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=legend_elements, loc="lower center", ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.suptitle("Test prediction spot-check (no ground truth)", fontsize=12)
plt.tight_layout()
plt.savefig(SUBMIT_DIR / f"spot_check_{run_date}.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {SUBMIT_DIR}/spot_check_{run_date}.png")

## 8. Submission Checklist

Before VB submits to CodaBench:

- [ ] Validator printed `OK` (cell 6)
- [ ] All 5 classes appear in the histogram (cell 4)
- [ ] Spot-check masks look spatially coherent (no pure noise)
- [ ] ZIP path: check `results/subtask1/submissions/submit_YYYYMMDD.zip`
- [ ] Submit at: https://www.codabench.org/competitions/12055/

After submission, VB reports the CodaBench score back so Claude can propose targeted improvements for the next iteration.

In [ ]:
print(f"Submission ZIP: {zip_path.resolve()}")
print(f"File size:      {zip_path.stat().st_size / 1e6:.1f} MB")
print(f"Patch count:    {len(mask_files)}")
print()
print("Config used:")
config = {
    "alpha": ALPHA,
    "median_filter_size": MEDIAN_FILTER_SIZE,
    "use_tta": USE_TTA,
    "temporal_option": TEMPORAL_OPTION,
    "pixel_model": PIXEL_MODEL_NAME,
}
print(json.dumps(config, indent=2))
(SUBMIT_DIR / f"submit_{run_date}_config.json").write_text(json.dumps(config, indent=2))
print(f"\nConfig saved → {SUBMIT_DIR}/submit_{run_date}_config.json")